# Montmory : optimisation TPU sans changer les bits

Choisir **Exécution → Modifier le type d’exécution → TPU** avant de commencer. Exécuter les quatre cellules de code dans l’ordre. Tous les fichiers sont intégrés au notebook. Aucun compte cloud payant n’est créé, aucun résultat n’est envoyé automatiquement.

26 configurations : 16 Montmory, 4 Threefry Pallas, 4 API JAX, RBG et témoin. Quatre volumes jusqu’à 128 Mio. Les finalistes sont figés avant la confirmation sur sept nouvelles seeds. Une sortie différente de la référence arrête le test.

## 1. Préparer les fichiers

In [ ]:
import base64, hashlib, io, os, pathlib, subprocess, sys, tarfile
ROOT = pathlib.Path(os.environ.get("MONTMORY_NOTEBOOK_ROOT", "/content")).resolve()
ROOT.mkdir(parents=True, exist_ok=True)
data = base64.b64decode('...')
assert hashlib.sha256(data).hexdigest() == 'a615a9c11c6c5378935090a552b7729efff17ce871aa719fdb1f600a08e1a372', "Archive incorrecte"
with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:
    for member in archive.getmembers():
        target = (ROOT / member.name).resolve()
        if not member.isfile() or not target.is_relative_to(ROOT / "montmory-tpu-opt"):
            raise RuntimeError("Entree d'archive invalide")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.extractfile(member).read())
KIT = ROOT / "montmory-tpu-opt"
subprocess.run([sys.executable, str(KIT / "check_manifest.py")], cwd=KIT, check=True,
               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Kit extrait et empreintes verifiees :", KIT)


## 2. Installer les versions prévues de JAX
Attendre la fin de l’installation.

In [ ]:
command = [sys.executable, "-m", "pip", "install", "-r", str(KIT / "requirements-tpu.txt")]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("Installation echouee : copier le message ci-dessus.")
print("Installation terminee. Passer a la cellule suivante.")


## 3. Vérifier, optimiser et confirmer sur TPU
Les messages s’affichent en direct. Laisser terminer les compilations et les trois blocs de confirmation. En cas d’erreur, exécuter quand même la cellule suivante pour télécharger les diagnostics.

In [ ]:
import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RESULTS = KIT / "results" / (stamp + "-tpu-opt")
RESULT_ARCHIVE = RESULTS.parent / (RESULTS.name + ".tar.gz")
command = [sys.executable, "-u", str(KIT / "run.py"), "--out", str(RESULTS)]
print("Demarrage : conformite, recherche, puis confirmation. Les compilations peuvent prendre plusieurs minutes.", flush=True)
with subprocess.Popen(command, cwd=KIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
print("Archive :", RESULT_ARCHIVE, flush=True)
if code:
    raise RuntimeError("Essai non valide. Executer la cellule de telechargement pour transmettre les diagnostics.")
print("Campagne terminee : resultats et sorties controles. Telecharger l'archive ci-dessous.")


## 4. Télécharger les résultats
Joindre ensuite le fichier `.tar.gz` à la conversation, ou le transférer sur le VPS.

In [ ]:
from google.colab import files
if not RESULT_ARCHIVE.exists():
    raise RuntimeError("Archive absente. Copier les dernieres lignes de l'erreur affichee.")
files.download(str(RESULT_ARCHIVE))


Les débits incluent le lancement et l’attente depuis Python. Ils excluent compilation, préparation des clés et transferts de contrôle. Une configuration n’est recommandée que pour les volumes, la génération TPU et la version JAX mesurés. La campagne vérifie les sorties entières ; elle ne lance pas de nouvelle batterie TestU01/TACM.